# 01 · PubMedQA data prep
**Purpose:** fixed stratified TEST (500) / DEV (500) split of the expert-labelled `pqa_labeled`, 5 DEV few-shot examples, and N unlabelled source abstracts for generation — with zero pubid overlap.
**Inputs:** HF `qiaojin/PubMedQA`. **Outputs:** `data/{test,dev,source}_pubids.json`, `data/fewshot.json`, `data/split_stats.json`; `artifacts/data/{test,dev,source_abstracts}.jsonl`.
**Runtime (T4):** ~2 min (CPU only).

In [ ]:
import os, sys, json, random
if not os.path.ismount("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/ps8/repo/src")
import ps8_common as C
C.ensure_env()
cfg = C.load_config(); FORCE = False
GPU, P = C.gpu_profile(cfg)
SEED = cfg["seed"]; N_SOURCE = cfg["data"]["n_source"] or P["n_source"]
OUT = C.ART / "data"; OUT.mkdir(parents=True, exist_ok=True)
print("profile", GPU, "| N_SOURCE", N_SOURCE)

In [ ]:
from collections import Counter
from datasets import load_dataset
labeled = load_dataset(cfg["data"]["hf_id"], "pqa_labeled", split="train")
unlabeled = load_dataset(cfg["data"]["hf_id"], "pqa_unlabeled", split="train")
print(labeled, unlabeled, sep="\n")
ex = labeled[0]
print({k: (v if k != "context" else {kk: str(vv)[:120] for kk, vv in v.items()}) for k, v in ex.items()})
print("pqa_labeled label distribution:", Counter(labeled["final_decision"]))

In [ ]:
def row(r, labelled=True):
    d = {"pubid": r["pubid"], "question": r["question"], "context_text": C.join_contexts(r["context"]),
         "long_answer": r["long_answer"]}
    if labelled:
        d["final_decision"] = r["final_decision"]
    return d

if FORCE or not (OUT / "test.jsonl").exists():
    from sklearn.model_selection import train_test_split
    idx = list(range(len(labeled)))
    test_idx, dev_idx = train_test_split(idx, test_size=len(labeled) - cfg["data"]["test_size"],
                                         stratify=labeled["final_decision"], random_state=SEED)
    test = [row(labeled[i]) for i in sorted(test_idx)]
    dev = [row(labeled[i]) for i in sorted(dev_idx)]
    C.write_jsonl(test, OUT / "test.jsonl"); C.write_jsonl(dev, OUT / "dev.jsonl")
test, dev = C.read_jsonl(OUT / "test.jsonl"), C.read_jsonl(OUT / "dev.jsonl")
test_ids, dev_ids = {r["pubid"] for r in test}, {r["pubid"] for r in dev}
assert len(test) == cfg["data"]["test_size"] and not (test_ids & dev_ids)
json.dump(sorted(test_ids), open(C.DATA / "test_pubids.json", "w"))
json.dump(sorted(dev_ids), open(C.DATA / "dev_pubids.json", "w"))
print("TEST", len(test), Counter(r["final_decision"] for r in test))
print("DEV ", len(dev), Counter(r["final_decision"] for r in dev))

In [ ]:
# 5 few-shot examples from DEV (2 yes, 2 no, 1 maybe), short abstracts' conclusions as reasoning style.
if FORCE or not (C.DATA / "fewshot.json").exists():
    rng = random.Random(SEED)
    fewshot = []
    for lab, k in (("yes", 2), ("no", 2), ("maybe", 1)):
        pool = [r for r in dev if r["final_decision"] == lab and len(r["long_answer"].split()) <= 60]
        fewshot += rng.sample(pool, k)
    json.dump(fewshot, open(C.DATA / "fewshot.json", "w"), indent=1)
fewshot = json.load(open(C.DATA / "fewshot.json"))
assert all(r["pubid"] in dev_ids for r in fewshot)
for r in fewshot:
    print(r["final_decision"], "|", r["question"])

In [ ]:
# N source abstracts from pqa_unlabeled; assert zero overlap with TEST and DEV.
if FORCE or not (OUT / "source_abstracts.jsonl").exists() or len(C.read_jsonl(OUT / "source_abstracts.jsonl")) != N_SOURCE:
    labelled_ids = test_ids | dev_ids
    cand = [i for i, p in enumerate(unlabeled["pubid"]) if p not in labelled_ids]
    pick = sorted(random.Random(SEED).sample(cand, N_SOURCE))
    C.write_jsonl([row(unlabeled[i], labelled=False) for i in pick], OUT / "source_abstracts.jsonl")
source = C.read_jsonl(OUT / "source_abstracts.jsonl")
src_ids = {r["pubid"] for r in source}
assert len(source) == N_SOURCE and not (src_ids & test_ids) and not (src_ids & dev_ids)
json.dump(sorted(src_ids), open(C.DATA / "source_pubids.json", "w"))
lens = sorted(len(r["context_text"].split()) for r in source)
print(f"SOURCE {len(source)} abstracts | words p50={lens[len(lens)//2]} p95={lens[int(.95*len(lens))]} max={lens[-1]}")

In [ ]:
stats = {"pqa_labeled": dict(Counter(labeled["final_decision"])),
         "test": dict(Counter(r["final_decision"] for r in test)), "dev": dict(Counter(r["final_decision"] for r in dev)),
         "n_source": len(source), "overlap_test_dev": 0, "overlap_source_labelled": 0, "seed": SEED}
json.dump(stats, open(C.DATA / "split_stats.json", "w"), indent=1)
print(json.dumps(stats, indent=1))
print("Saved:", OUT, C.DATA)